# Chapter 3.6: Semantic ID Methods

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain** why semantic IDs are needed: from opaque item IDs to meaningful tokens
2. **Implement** VQ-VAE for learning discrete item codebooks
3. **Build** RQ-VAE (Residual Quantization VAE) for hierarchical semantic codes
4. **Understand** hierarchical tokenization: coarse-to-fine item representation
5. **Compare** atomic IDs, semantic IDs, and content-based representations
6. **Analyze** codebook utilization and quality metrics
7. **Apply** semantic IDs for structured item indexing

## Prerequisites

- Embedding-based retrieval (Chapter 3.1)
- Basic understanding of autoencoders
- PyTorch fundamentals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part3/chapter_3.6_semantic_ids.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/raw/main/notebooks/part3/chapter_3.6_semantic_ids.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cpu')
print(f"PyTorch version: {torch.__version__}")

## 1. Why Semantic IDs?

Traditional recommendation systems use **atomic IDs**: each item gets a unique integer (e.g., item #47293). Problems:

1. **No generalization**: Item 47293 and 47294 share no structure, even if they're similar products
2. **Cold start**: New items have no learned embedding
3. **Scaling**: Vocabulary grows linearly with catalog size
4. **No compositionality**: Can't compose or decompose item representations

**Semantic IDs** replace opaque integers with meaningful token sequences:

$$\text{Item} \rightarrow (c_1, c_2, \ldots, c_L)$$

where each $c_l$ is a code from a codebook $\mathcal{C}_l$ of size $K$.

This creates a hierarchical structure:
- $c_1$: coarse category (e.g., "Electronics")
- $c_2$: subcategory (e.g., "Phones")
- $c_3$: fine-grained (e.g., "iPhone-like")

> **💡 Concept:** Semantic IDs make items compositional. Two items sharing the first code are in the same coarse category. This enables tree-structured search and generalization to new items.

In [ ]:
# Demonstrate the limitation of atomic IDs
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Atomic IDs: random embedding, no structure
np.random.seed(42)
n_items = 200
categories = np.repeat(np.arange(5), n_items // 5)
colors_map = plt.cm.Set1(np.linspace(0, 0.8, 5))
cat_names = ['Electronics', 'Books', 'Sports', 'Fashion', 'Food']

# Random embeddings (like untrained atomic IDs)
random_emb = np.random.randn(n_items, 2)
for cat_id in range(5):
    mask = categories == cat_id
    axes[0].scatter(random_emb[mask, 0], random_emb[mask, 1], 
                    c=[colors_map[cat_id]], s=20, alpha=0.6, label=cat_names[cat_id])
axes[0].set_title('Atomic IDs (No Structure)', fontsize=13)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Semantic IDs: structured embedding
structured_emb = np.zeros((n_items, 2))
for cat_id in range(5):
    mask = categories == cat_id
    center = np.array([np.cos(2 * np.pi * cat_id / 5), np.sin(2 * np.pi * cat_id / 5)]) * 3
    structured_emb[mask] = center + np.random.randn(mask.sum(), 2) * 0.5

for cat_id in range(5):
    mask = categories == cat_id
    axes[1].scatter(structured_emb[mask, 0], structured_emb[mask, 1],
                    c=[colors_map[cat_id]], s=20, alpha=0.6, label=cat_names[cat_id])
axes[1].set_title('Semantic IDs (Structured)', fontsize=13)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Atomic IDs vs Semantic IDs in Embedding Space', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. VQ-VAE: Vector Quantized Variational Autoencoder

VQ-VAE (van den Oord et al., 2017) learns a discrete codebook by:

1. **Encode** input to continuous embedding: $\mathbf{z}_e = f_{\text{enc}}(\mathbf{x})$
2. **Quantize** to nearest codebook entry: $\mathbf{z}_q = \arg\min_{\mathbf{e}_k} \|\mathbf{z}_e - \mathbf{e}_k\|$
3. **Decode** from quantized embedding: $\hat{\mathbf{x}} = f_{\text{dec}}(\mathbf{z}_q)$

### Loss Function

$$\mathcal{L} = \underbrace{\|\mathbf{x} - \hat{\mathbf{x}}\|^2}_{\text{reconstruction}} + \underbrace{\|\text{sg}[\mathbf{z}_e] - \mathbf{e}\|^2}_{\text{codebook}} + \beta \underbrace{\|\mathbf{z}_e - \text{sg}[\mathbf{e}]\|^2}_{\text{commitment}}$$

where $\text{sg}[\cdot]$ is the stop-gradient operator.

> **⚠️ Common Pitfall:** VQ-VAE codebooks often suffer from **index collapse** — only a few codes get used. Solutions include EMA updates, codebook reset, and straight-through estimation with temperature.

In [ ]:
class VectorQuantizer(nn.Module):
    """Vector Quantization layer with EMA codebook update."""
    
    def __init__(self, num_codes: int, code_dim: int, 
                 commitment_cost: float = 0.25, decay: float = 0.99):
        super().__init__()
        self.num_codes = num_codes
        self.code_dim = code_dim
        self.commitment_cost = commitment_cost
        self.decay = decay
        
        # Codebook
        self.codebook = nn.Embedding(num_codes, code_dim)
        nn.init.uniform_(self.codebook.weight, -1/num_codes, 1/num_codes)
        
        # EMA tracking
        self.register_buffer('ema_count', torch.zeros(num_codes))
        self.register_buffer('ema_weight', self.codebook.weight.clone())
    
    def forward(self, z_e: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            z_e: (B, D) continuous embeddings from encoder
        
        Returns:
            z_q: (B, D) quantized embeddings
            indices: (B,) codebook indices
            loss: VQ loss
        """
        # Compute distances to codebook entries
        distances = torch.cdist(z_e.unsqueeze(0), self.codebook.weight.unsqueeze(0)).squeeze(0)
        
        # Find nearest codes
        indices = distances.argmin(dim=-1)  # (B,)
        z_q = self.codebook(indices)  # (B, D)
        
        # EMA update (during training)
        if self.training:
            with torch.no_grad():
                one_hot = F.one_hot(indices, self.num_codes).float()  # (B, K)
                self.ema_count = self.decay * self.ema_count + (1 - self.decay) * one_hot.sum(0)
                dw = one_hot.T @ z_e  # (K, D)
                self.ema_weight = self.decay * self.ema_weight + (1 - self.decay) * dw
                
                # Laplace smoothing
                n = self.ema_count.sum()
                count = (self.ema_count + 1e-5) / (n + self.num_codes * 1e-5) * n
                self.codebook.weight.data = self.ema_weight / count.unsqueeze(1)
        
        # Losses
        codebook_loss = F.mse_loss(z_e.detach(), z_q)
        commitment_loss = F.mse_loss(z_e, z_q.detach())
        loss = codebook_loss + self.commitment_cost * commitment_loss
        
        # Straight-through estimator
        z_q = z_e + (z_q - z_e).detach()
        
        return z_q, indices, loss


class VQVAE(nn.Module):
    """VQ-VAE for item tokenization."""
    
    def __init__(self, input_dim: int, hidden_dim: int = 64, 
                 code_dim: int = 32, num_codes: int = 256):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, code_dim)
        )
        
        self.vq = VectorQuantizer(num_codes, code_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(code_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def forward(self, x: torch.Tensor):
        z_e = self.encoder(x)
        z_q, indices, vq_loss = self.vq(z_e)
        x_hat = self.decoder(z_q)
        recon_loss = F.mse_loss(x_hat, x)
        return x_hat, indices, recon_loss + vq_loss
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Get semantic ID (codebook index) for items."""
        z_e = self.encoder(x)
        _, indices, _ = self.vq(z_e)
        return indices


# Create synthetic item features
NUM_ITEMS = 5000
FEATURE_DIM = 64
NUM_CATEGORIES = 10

# Items have category-dependent features
item_categories = np.random.randint(0, NUM_CATEGORIES, NUM_ITEMS)
category_centers = np.random.randn(NUM_CATEGORIES, FEATURE_DIM).astype(np.float32) * 2
item_features = np.array([
    category_centers[cat] + np.random.randn(FEATURE_DIM).astype(np.float32) * 0.5
    for cat in item_categories
])
item_features_tensor = torch.tensor(item_features)

print(f"Item features shape: {item_features.shape}")
print(f"Categories: {NUM_CATEGORIES}")

In [ ]:
# Train VQ-VAE
vqvae = VQVAE(FEATURE_DIM, hidden_dim=64, code_dim=32, num_codes=64)
optimizer = torch.optim.Adam(vqvae.parameters(), lr=1e-3)

losses = []
vqvae.train()

for epoch in range(100):
    # Random mini-batch
    idx = np.random.choice(NUM_ITEMS, 256, replace=False)
    batch = item_features_tensor[idx]
    
    x_hat, indices, loss = vqvae(batch)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if (epoch + 1) % 25 == 0:
        # Check codebook utilization
        with torch.no_grad():
            all_indices = vqvae.encode(item_features_tensor)
            unique_codes = len(torch.unique(all_indices))
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | "
              f"Active codes: {unique_codes}/64")

plt.figure(figsize=(8, 4))
plt.plot(losses, color='coral', linewidth=1.5, alpha=0.8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('VQ-VAE Loss', fontsize=12)
plt.title('VQ-VAE Training', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. RQ-VAE: Residual Quantization VAE

RQ-VAE (Zeghidour et al., 2021; Lee et al., 2022) applies **residual quantization** — quantize the residual error at each level:

1. Level 1: $\hat{\mathbf{z}}_1 = Q_1(\mathbf{z}_e)$, residual $\mathbf{r}_1 = \mathbf{z}_e - \hat{\mathbf{z}}_1$
2. Level 2: $\hat{\mathbf{z}}_2 = Q_2(\mathbf{r}_1)$, residual $\mathbf{r}_2 = \mathbf{r}_1 - \hat{\mathbf{z}}_2$
3. Level $L$: $\hat{\mathbf{z}}_L = Q_L(\mathbf{r}_{L-1})$

The semantic ID becomes $(c_1, c_2, \ldots, c_L)$ where $c_l$ is the codebook index at level $l$.

Reconstruction: $\hat{\mathbf{z}} = \sum_{l=1}^{L} \hat{\mathbf{z}}_l$

This creates a **hierarchical** representation:
- $c_1$ captures coarse structure
- $c_2$ refines $c_1$
- $c_L$ captures fine details

> **💡 Concept:** RQ-VAE is the key building block of TIGER (Google, 2023). Each item gets a semantic ID like (23, 156, 42), which can be generated autoregressively by a language model.

In [ ]:
class ResidualVectorQuantizer(nn.Module):
    """Residual Vector Quantization with multiple codebook levels."""
    
    def __init__(self, num_levels: int, num_codes: int, code_dim: int,
                 commitment_cost: float = 0.25):
        super().__init__()
        self.num_levels = num_levels
        self.num_codes = num_codes
        
        # One codebook per level
        self.quantizers = nn.ModuleList([
            VectorQuantizer(num_codes, code_dim, commitment_cost)
            for _ in range(num_levels)
        ])
    
    def forward(self, z: torch.Tensor) -> Tuple[torch.Tensor, List[torch.Tensor], torch.Tensor]:
        """
        Args:
            z: (B, D) continuous embedding
        
        Returns:
            z_q: (B, D) sum of all quantized residuals
            all_indices: list of (B,) code indices per level
            total_loss: sum of VQ losses across levels
        """
        residual = z
        z_q_sum = torch.zeros_like(z)
        all_indices = []
        total_loss = 0.0
        
        for level, quantizer in enumerate(self.quantizers):
            z_q_level, indices, loss = quantizer(residual)
            z_q_sum = z_q_sum + z_q_level
            residual = residual - z_q_level.detach()  # Quantize the residual
            all_indices.append(indices)
            total_loss = total_loss + loss
        
        return z_q_sum, all_indices, total_loss
    
    def encode(self, z: torch.Tensor) -> List[torch.Tensor]:
        """Get multi-level semantic IDs."""
        residual = z
        all_indices = []
        
        for quantizer in self.quantizers:
            z_q, indices, _ = quantizer(residual)
            all_indices.append(indices)
            residual = residual - z_q
        
        return all_indices


class RQVAE(nn.Module):
    """RQ-VAE for generating hierarchical semantic item IDs."""
    
    def __init__(self, input_dim: int, hidden_dim: int = 64, code_dim: int = 32,
                 num_levels: int = 3, num_codes: int = 256):
        super().__init__()
        self.num_levels = num_levels
        self.num_codes = num_codes
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, code_dim)
        )
        
        self.rq = ResidualVectorQuantizer(num_levels, num_codes, code_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(code_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def forward(self, x: torch.Tensor):
        z_e = self.encoder(x)
        z_q, indices, vq_loss = self.rq(z_e)
        x_hat = self.decoder(z_q)
        recon_loss = F.mse_loss(x_hat, x)
        return x_hat, indices, recon_loss + vq_loss
    
    def get_semantic_ids(self, x: torch.Tensor) -> torch.Tensor:
        """Get semantic IDs as (B, num_levels) tensor."""
        z_e = self.encoder(x)
        all_indices = self.rq.encode(z_e)
        return torch.stack(all_indices, dim=1)  # (B, L)


# Build RQ-VAE
rqvae = RQVAE(FEATURE_DIM, hidden_dim=64, code_dim=32, 
              num_levels=3, num_codes=64)
print(f"RQ-VAE parameters: {sum(p.numel() for p in rqvae.parameters()):,}")
print(f"Semantic ID space: {64}^{3} = {64**3:,} possible IDs")

In [ ]:
# Train RQ-VAE
optimizer = torch.optim.Adam(rqvae.parameters(), lr=1e-3)

losses = []
rqvae.train()

for epoch in range(150):
    idx = np.random.choice(NUM_ITEMS, 256, replace=False)
    batch = item_features_tensor[idx]
    
    x_hat, indices, loss = rqvae(batch)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# Evaluate
rqvae.eval()
with torch.no_grad():
    semantic_ids = rqvae.get_semantic_ids(item_features_tensor)  # (N, 3)

print(f"\nSemantic IDs shape: {semantic_ids.shape}")
print(f"Example semantic IDs (first 5 items):")
for i in range(5):
    sid = semantic_ids[i].tolist()
    print(f"  Item {i} (category {item_categories[i]}): ({sid[0]}, {sid[1]}, {sid[2]})")

# Codebook utilization per level
for level in range(3):
    unique_codes = len(torch.unique(semantic_ids[:, level]))
    print(f"\nLevel {level+1}: {unique_codes}/64 codes used ({unique_codes/64*100:.0f}% utilization)")

In [ ]:
# Visualize: do items with same level-1 code share categories?
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for level in range(3):
    # For each code, compute category distribution
    codes = semantic_ids[:, level].numpy()
    unique_codes_l = np.unique(codes)
    
    # Compute purity: what fraction of items with the same code share the same category?
    purities = []
    code_sizes = []
    for code in unique_codes_l:
        mask = codes == code
        if mask.sum() < 3:
            continue
        cats = item_categories[mask]
        most_common_frac = np.max(np.bincount(cats)) / len(cats)
        purities.append(most_common_frac)
        code_sizes.append(mask.sum())
    
    axes[level].hist(purities, bins=20, color=f'C{level}', alpha=0.7, edgecolor='white')
    axes[level].set_xlabel('Category Purity', fontsize=11)
    axes[level].set_ylabel('Count', fontsize=11)
    axes[level].set_title(f'Level {level+1} Code Purity\n(avg={np.mean(purities):.2f})', fontsize=12)
    axes[level].axvline(x=1.0/NUM_CATEGORIES, color='red', linestyle='--', 
                        label=f'Random baseline ({1/NUM_CATEGORIES:.2f})')
    axes[level].legend(fontsize=8)

plt.suptitle('RQ-VAE Code Purity (How well codes align with categories)', fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

# Training loss
plt.figure(figsize=(8, 4))
plt.plot(losses, color='darkorange', linewidth=1.5, alpha=0.8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('RQ-VAE Loss', fontsize=12)
plt.title('RQ-VAE Training', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Hierarchical Tokenization: Coarse-to-Fine

The hierarchical structure of RQ-VAE IDs enables efficient tree-structured search:

```
Level 1 (64 codes):  [23] → Electronics cluster
Level 2 (64 codes):  [23, 7] → Phone subcategory
Level 3 (64 codes):  [23, 7, 42] → Specific phone type
```

### Comparison: ID Representations

| Representation | Vocab Size | Generalization | Compositionality |
|---------------|-----------|----------------|------------------|
| Atomic ID | $N$ items | None | None |
| Semantic ID (RQ-VAE) | $K^L$ (e.g., $256^3$) | Within codes | Hierarchical |
| Content-based | Continuous | Full | Via similarity |

> **🔑 Pro Tip:** The choice of codebook size and number of levels involves a trade-off. More levels = more precise IDs but longer sequences for generative models. Typical choices: 3-4 levels, 256-1024 codes per level.

In [ ]:
# Demonstrate hierarchical retrieval
def hierarchical_search(query_semantic_id, all_semantic_ids, item_features, 
                        query_features, levels_to_match=1):
    """Search by matching semantic ID prefixes."""
    # Match at specified level
    prefix = query_semantic_id[:levels_to_match]
    all_prefixes = all_semantic_ids[:, :levels_to_match]
    
    # Find items with matching prefix
    match_mask = (all_prefixes == prefix.unsqueeze(0)).all(dim=1)
    matching_indices = torch.where(match_mask)[0]
    
    if len(matching_indices) == 0:
        return torch.tensor([]), 0
    
    # Re-rank by actual feature similarity
    matching_features = item_features[matching_indices]
    similarities = F.cosine_similarity(query_features.unsqueeze(0), matching_features)
    sorted_idx = similarities.argsort(descending=True)
    
    return matching_indices[sorted_idx], len(matching_indices)

# Demo
query_idx = 0
query_sid = semantic_ids[query_idx]
query_feat = item_features_tensor[query_idx]

print(f"Query item {query_idx}: category={item_categories[query_idx]}, " 
      f"semantic ID=({query_sid[0].item()}, {query_sid[1].item()}, {query_sid[2].item()})")

for levels in [1, 2, 3]:
    results, n_candidates = hierarchical_search(
        query_sid, semantic_ids, item_features_tensor, query_feat, levels
    )
    if n_candidates > 0:
        result_cats = item_categories[results[:10].numpy()]
        same_cat_frac = (result_cats == item_categories[query_idx]).mean()
        print(f"  Level 1-{levels} match: {n_candidates:5d} candidates, "
              f"top-10 same-category rate: {same_cat_frac:.2f}")

## 5. Codebook Quality Analysis

In [ ]:
# Analyze reconstruction quality
rqvae.eval()
with torch.no_grad():
    x_hat, _, _ = rqvae(item_features_tensor)
    recon_error = F.mse_loss(x_hat, item_features_tensor, reduction='none').mean(dim=1)

# Also compute VQ-VAE reconstruction for comparison
vqvae.eval()
with torch.no_grad():
    x_hat_vq, _, _ = vqvae(item_features_tensor)
    recon_error_vq = F.mse_loss(x_hat_vq, item_features_tensor, reduction='none').mean(dim=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(recon_error_vq.numpy(), bins=50, alpha=0.6, label='VQ-VAE (1 level)', color='steelblue')
axes[0].hist(recon_error.numpy(), bins=50, alpha=0.6, label='RQ-VAE (3 levels)', color='coral')
axes[0].set_xlabel('Reconstruction Error (MSE)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Reconstruction Quality', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Codebook usage distribution
level1_codes = semantic_ids[:, 0].numpy()
code_counts = np.bincount(level1_codes, minlength=64)
sorted_counts = np.sort(code_counts)[::-1]

axes[1].bar(range(len(sorted_counts)), sorted_counts, color='teal', alpha=0.7)
axes[1].set_xlabel('Code Rank', fontsize=11)
axes[1].set_ylabel('Items per Code', fontsize=11)
axes[1].set_title('Level 1 Codebook Usage Distribution', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"VQ-VAE avg reconstruction error: {recon_error_vq.mean():.4f}")
print(f"RQ-VAE avg reconstruction error: {recon_error.mean():.4f}")
print(f"Improvement: {(1 - recon_error.mean() / recon_error_vq.mean()) * 100:.1f}%")

## Exercises

### 🏋️ Exercise 1: Implement RQ-VAE with Codebook Reset

Dead codes (unused codebook entries) waste capacity. Implement a codebook reset mechanism that re-initializes dead codes from the training data.

In [ ]:
class VectorQuantizerWithReset(nn.Module):
    """VQ with dead code reset."""
    def __init__(self, num_codes, code_dim, reset_threshold=1):
        super().__init__()
        # TODO: Track usage count per code
        # TODO: After N training steps, reset codes with
        #       usage < reset_threshold by reinitializing from
        #       training data samples
        pass
    
    def forward(self, z_e):
        # TODO: Standard VQ forward
        # TODO: Periodically check and reset dead codes
        pass

### 🏋️ Exercise 2: Evaluate Semantic ID Quality

Measure how well the semantic IDs preserve item similarities.

In [ ]:
def evaluate_semantic_id_quality(semantic_ids, item_features, item_categories,
                                  k=10):
    """
    Evaluate semantic ID quality with multiple metrics:
    
    1. Category purity: items sharing the same ID prefix should share categories
    2. Similarity preservation: items with similar features should have similar IDs
    3. Reconstruction quality: decoded features should match original
    4. Codebook utilization: fraction of codes actively used
    
    Returns:
        Dict of metrics
    """
    # TODO: Compute category purity at each level
    # TODO: Compute rank correlation between feature similarity and ID overlap
    # TODO: Compute codebook utilization
    # TODO: Visualize results
    pass

### 🏋️ Exercise 3: Implement Gumbel-Softmax VQ-VAE

Replace the hard argmin in VQ with Gumbel-Softmax for differentiable discrete sampling.

In [ ]:
class GumbelVectorQuantizer(nn.Module):
    """VQ using Gumbel-Softmax for differentiable quantization."""
    def __init__(self, num_codes, code_dim, temperature=1.0):
        super().__init__()
        # TODO: Define codebook and projection to logits
        # TODO: Implement Gumbel-Softmax selection
        # TODO: Anneal temperature during training
        pass
    
    def forward(self, z_e, hard=True):
        # TODO: Project z_e to logits over codebook
        # TODO: Apply Gumbel-Softmax (hard in forward, soft in backward)
        # TODO: Return quantized embedding and indices
        pass

# Hint: Use torch.nn.functional.gumbel_softmax

## Summary

| Method | Codes per Item | Hierarchy | Training | Used By |
|--------|---------------|-----------|----------|---------|
| **Atomic ID** | 1 (opaque) | None | Embedding table | Traditional RecSys |
| **VQ-VAE** | 1 (learned) | Single level | Reconstruction | - |
| **RQ-VAE** | L (hierarchical) | Multi-level | Residual reconstruction | TIGER, DSI |
| **Content hash** | L (deterministic) | Fixed | None (unsupervised) | Simple baselines |

**Key takeaway**: Semantic IDs bridge the gap between discrete item catalogs and continuous embedding spaces. They enable generative retrieval (Chapter 3.7) by making items "speakable" in a token vocabulary.

**Next up**: Chapter 3.7 covers Generative Retrieval — using language model-style autoregressive generation to retrieve items by generating their semantic IDs.